In [1]:
import sympy as sp

In [ ]:
# Symbolic PDE operator on
# y(t, σ, θ) = exp( α(t) + β(t)^T [σ, θ] + 1/2 [σ, θ]^T Γ(t) [σ, θ] )
#
# Requirements addressed:
# - Pretty-print with true Greek symbols (α, β, Γ, κ, η, λ, θ₀, σ, θ).
# - Output collected by monomials in σ and θ (1, σ, θ, σ², σθ, θ²).

sp.init_printing(use_unicode=True)

# ----- Time & state vars (use Greek symbol names for pretty output) -----
t   = sp.symbols('t', real=True)
σ, θ = sp.symbols('σ θ', real=True)

# ----- Parameters (pretty names; Python variables can be ASCII) -----
η     = sp.Symbol('η', real=True)
θ0    = sp.Symbol('θ₀', real=True)   # displayed as θ₀ in output
λ     = sp.Symbol('λ', real=True)    # displayed as λ
κ     = sp.Symbol('κ', real=True)    # displayed as κ
s1    = sp.Symbol('s₁', real=True)
s2    = sp.Symbol('s₂', real=True)

# ----- Time-dependent coefficients with Greek names -----
α   = sp.Function('α')(t)

β1  = sp.Function('β₁')(t)
β2  = sp.Function('β₂')(t)

Γ11 = sp.Function('Γ₁₁')(t)
Γ12 = sp.Function('Γ₁₂')(t)   # = Γ21 by symmetry
Γ22 = sp.Function('Γ₂₂')(t)

# Vectors/matrices
V     = sp.Matrix([σ, θ])
β     = sp.Matrix([β1, β2])
Γ     = sp.Matrix([[Γ11, Γ12],
                   [Γ12, Γ22]])

# ----- y definition -----
quad = α + (β.T * V)[0] + sp.Rational(1, 2) * (V.T * Γ * V)[0]
y = sp.exp(quad)

# ----- Derivatives -----
y_t    = sp.diff(y, t)
y_σ    = sp.diff(y, σ)
y_σσ   = sp.diff(y, σ, 2)
y_θ    = sp.diff(y, θ)
y_θθ   = sp.diff(y, θ, 2)

# ----- Operator L applied to y -----
# L[y] = y_t + (1/2) σ^2 y_{σσ} + (1/2) (η − θ₀ − λ t + θ)^2 y_{θθ}
#        + κ(θ − σ) y_σ + λ y_θ − (s₁ σ^2 + s₂ σ θ) y
L_y = (
    y_t
    + sp.Rational(1, 2) * σ**2 * y_σσ
    + sp.Rational(1, 2) * (η - θ0 - λ*t + θ)**2 * y_θθ
    + κ * (θ - σ) * y_σ
    + λ * y_θ
    - (s1 * σ**2 + s2 * σ * θ) * y
)

# ----- Factor out y to expose exponential-affine polynomial -----
P = sp.simplify(sp.expand(L_y / y))  # P(σ, θ; t) so that L[y] = y * P

In [8]:
y

In [9]:
L_y

In [3]:
# Canonical bivariate polynomial view in (σ, θ)
Poly_P = sp.Poly(sp.expand(P), σ, θ)

# Rebuild P as a sum of monomials σ^i θ^j * c_ij (this cleanly captures all interaction terms)
# Optional: sort by total degree, then lex in (i, j) for stable pretty output
monoms = Poly_P.monoms()
coeffs_list = Poly_P.coeffs()
ordering = sorted(
    zip(monoms, coeffs_list),
    key=lambda mj: (mj[0][0] + mj[0][1], mj[0][0], mj[0][1])  # (total degree, i, j)
)
P_collected = sp.Add(*[c * σ**i * θ**j for (i, j), c in ordering])
P_collected



In [4]:
y

In [5]:
L_y

In [6]:
# If you still want quick coefficient lookup for specific monomials:
coeff_map = dict(zip(monoms, coeffs_list))
c_00 = coeff_map.get((0, 0), sp.Integer(0))
c_10 = coeff_map.get((1, 0), sp.Integer(0))
c_01 = coeff_map.get((0, 1), sp.Integer(0))
c_20 = coeff_map.get((2, 0), sp.Integer(0))
c_11 = coeff_map.get((1, 1), sp.Integer(0))
c_02 = coeff_map.get((0, 2), sp.Integer(0))


print("\nSelected coefficients:")
print("  [σ^0 θ^0]:", c_00)
print("  [σ^1 θ^0]:", c_10)
print("  [σ^0 θ^1]:", c_01)
print("  [σ^2 θ^0]:", c_20)
print("  [σ^1 θ^1]:", c_11)
print("  [σ^0 θ^2]:", c_02)



Selected coefficients:
  [σ^0 θ^0]: t**2*λ**2*Γ₂₂(t)/2 + t**2*λ**2*β₂(t)**2/2 - t*η*λ*Γ₂₂(t) - t*η*λ*β₂(t)**2 + t*θ₀*λ*Γ₂₂(t) + t*θ₀*λ*β₂(t)**2 + η**2*Γ₂₂(t)/2 + η**2*β₂(t)**2/2 - η*θ₀*Γ₂₂(t) - η*θ₀*β₂(t)**2 + θ₀**2*Γ₂₂(t)/2 + θ₀**2*β₂(t)**2/2 + λ*β₂(t) + Derivative(α(t), t)
  [σ^1 θ^0]: t**2*λ**2*Γ₁₂(t)*β₂(t) - 2*t*η*λ*Γ₁₂(t)*β₂(t) + 2*t*θ₀*λ*Γ₁₂(t)*β₂(t) + η**2*Γ₁₂(t)*β₂(t) - 2*η*θ₀*Γ₁₂(t)*β₂(t) + θ₀**2*Γ₁₂(t)*β₂(t) - κ*β₁(t) + λ*Γ₁₂(t) + Derivative(β₁(t), t)
  [σ^0 θ^1]: t**2*λ**2*Γ₂₂(t)*β₂(t) - 2*t*η*λ*Γ₂₂(t)*β₂(t) + 2*t*θ₀*λ*Γ₂₂(t)*β₂(t) - t*λ*Γ₂₂(t) - t*λ*β₂(t)**2 + η**2*Γ₂₂(t)*β₂(t) - 2*η*θ₀*Γ₂₂(t)*β₂(t) + η*Γ₂₂(t) + η*β₂(t)**2 + θ₀**2*Γ₂₂(t)*β₂(t) - θ₀*Γ₂₂(t) - θ₀*β₂(t)**2 + κ*β₁(t) + λ*Γ₂₂(t) + Derivative(β₂(t), t)
  [σ^2 θ^0]: -s₁ + t**2*λ**2*Γ₁₂(t)**2/2 - t*η*λ*Γ₁₂(t)**2 + t*θ₀*λ*Γ₁₂(t)**2 + η**2*Γ₁₂(t)**2/2 - η*θ₀*Γ₁₂(t)**2 + θ₀**2*Γ₁₂(t)**2/2 - κ*Γ₁₁(t) + Γ₁₁(t)/2 + β₁(t)**2/2 + Derivative(Γ₁₁(t), t)/2
  [σ^1 θ^1]: -s₂ + t**2*λ**2*Γ₁₂(t)*Γ₂₂(t) - 2*t*η*λ*Γ₁₂(t)*Γ₂₂(t) + 2

In [ ]:
# Display coefficients aligned with monomials
print("\nMonomial coefficients in P(σ, θ; t):")
print("  1      :", coeffs[(0, 0)])
print("  σ      :", coeffs[(1, 0)])
print("  θ      :", coeffs[(0, 1)])
print("  σ²     :", coeffs[(2, 0)])
print("  σ·θ    :", coeffs[(1, 1)])
print("  θ²     :", coeffs[(0, 2)])